# Bank Customer Churn – Classification & Explainable AI (XAI)

**Course:** 2400-DS2ML2 – Machine Learning 2  
**Authors:** Ronald Mjonono (473522) · Linxiao Mu (474569)

### Pipeline Overview
1. Exploratory Data Analysis (EDA)
2. Preprocessing and train–test split
3. Decision Tree classifier
4. Random Forest
5. Gradient Boosting
6. Neural Network (MLP)
7. Model evaluation & comparison
8. Baseline feature importance
9. XAI – SHAP global analysis
10. XAI – SHAP local explanations
11. XAI – SHAP dependence plots
12. XAI – SHAP cross-model comparison
13. XAI – LIME local explanations
14. XAI – Business segmentation
15. Conclusions

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import shap
import lime
import lime.lime_tabular

shap.initjs()
pd.set_option("display.max_columns", None)
sns.set_theme()

print("All libraries imported successfully.")

## 1. Data Loading and Initial Inspection

In [ ]:
df = pd.read_csv("../data/Churn_Modelling.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
id_like_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
print("Dropping:", id_like_cols)
df = df.drop(columns=id_like_cols)
df.head(3)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
target_col = "Exited"
print(df[target_col].value_counts(normalize=True).rename("proportion"))

sns.countplot(x=target_col, data=df)
plt.title("Target distribution (Exited)")
plt.show()


In [ ]:
numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()

if target_col in numeric_features:
    numeric_features.remove(target_col)
if target_col in categorical_features:
    categorical_features.remove(target_col)

print("Numeric features   :", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
df[numeric_features].hist(bins=30, figsize=(15, 10))
plt.suptitle("Histograms of numeric features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(5, 4))
    sns.countplot(x=col, data=df)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
or col in categorical_features:
    plt.figure(figsize=(5, 4))
    churn_rate = df.groupby(col)[target_col].mean()
    sns.barplot(x=churn_rate.index, y=churn_rate.values)
    plt.title(f"Churn rate by {col}")
    plt.ylabel("Mean Exited")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_features[:6]:
    plt.figure(figsize=(5, 4))
    sns.boxplot(x=target_col, y=col, data=df)
    plt.title(f"{col} by Exited")
    plt.tight_layout()
    plt.show()

In [ ]:
corr = df[numeric_features + [target_col]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

## 3. Train–Test Split and Preprocessing

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree_preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

nn_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

## 4. Evaluation Helper Function

In [ ]:
def evaluate_model(y_true, y_pred, y_proba=None, model_name="Model"):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan

    print(f"=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC AUC: {auc:.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, zero_division=0))

    return {"model": model_name, "accuracy": acc, "precision": prec,
            "recall": rec, "f1": f1, "roc_auc": auc}

## 5. Decision Tree Classifier

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dtree = Pipeline([
    ("preprocess", tree_preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])

param_grid_tree = {
    "clf__max_depth"        : [3, 5, 8, None],
    "clf__min_samples_split": [2, 10, 20],
    "clf__min_samples_leaf" : [1, 5, 10],
    "clf__criterion"        : ["gini", "entropy"]
}

print("Training Decision Tree...")
grid_tree = GridSearchCV(dtree, param_grid_tree, cv=cv, scoring="roc_auc", n_jobs=1)
grid_tree.fit(X_train, y_train)
print("Best params:", grid_tree.best_params_)

best_tree    = grid_tree.best_estimator_
y_pred_tree  = best_tree.predict(X_test)
y_proba_tree = best_tree.predict_proba(X_test)[:, 1]
tree_results = evaluate_model(y_test, y_pred_tree, y_proba_tree, "Decision Tree")

## 6. Random Forest

In [ ]:
rf = Pipeline([
    ("preprocess", tree_preprocessor),
    ("clf", RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    "clf__n_estimators"     : [100, 200],
    "clf__max_depth"        : [None, 6, 10],
    "clf__max_features"     : ["sqrt", "log2"],
    "clf__min_samples_split": [2, 10]
}

print("Training Random Forest...")
grid_rf = GridSearchCV(rf, param_grid_rf, cv=cv, scoring="roc_auc", n_jobs=1)
grid_rf.fit(X_train, y_train)
print("Best params:", grid_rf.best_params_)

best_rf    = grid_rf.best_estimator_
y_pred_rf  = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
rf_results = evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")

## 7. Gradient Boosting

In [ ]:
gb = Pipeline([
    ("preprocess", tree_preprocessor),
    ("clf", GradientBoostingClassifier(random_state=42))
])

param_grid_gb = {
    "clf__n_estimators" : [100, 200],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth"    : [2, 3],
    "clf__subsample"    : [0.8, 1.0]
}

print("Training Gradient Boosting...")
grid_gb = GridSearchCV(gb, param_grid_gb, cv=cv, scoring="roc_auc", n_jobs=1)
grid_gb.fit(X_train, y_train)
print("Best params:", grid_gb.best_params_)

best_gb    = grid_gb.best_estimator_
y_pred_gb  = best_gb.predict(X_test)
y_proba_gb = best_gb.predict_proba(X_test)[:, 1]
gb_results = evaluate_model(y_test, y_pred_gb, y_proba_gb, "Gradient Boosting")

## 8. Neural Network (MLP)

In [ ]:
mlp = Pipeline([
    ("preprocess", nn_preprocessor),
    ("clf", MLPClassifier(max_iter=200, random_state=42))
])

param_grid_mlp = {
    "clf__hidden_layer_sizes": [(32,), (64, 32)],
    "clf__alpha"             : [0.0001, 0.001],
    "clf__learning_rate_init": [0.001, 0.01]
}

print("Training MLP...")
grid_mlp = GridSearchCV(mlp, param_grid_mlp, cv=cv, scoring="roc_auc", n_jobs=1)
grid_mlp.fit(X_train, y_train)
print("Best params:", grid_mlp.best_params_)

best_mlp    = grid_mlp.best_estimator_
y_pred_mlp  = best_mlp.predict(X_test)
y_proba_mlp = best_mlp.predict_proba(X_test)[:, 1]
mlp_results = evaluate_model(y_test, y_pred_mlp, y_proba_mlp, "Neural Network (MLP)")

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame([tree_results, rf_results, gb_results, mlp_results])
results_df.set_index("model", inplace=True)
print(results_df.round(4))


In [ ]:
plt.figure(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_test, y_proba_tree, name="Decision Tree", ax=plt.gca())
RocCurveDisplay.from_predictions(y_test, y_proba_rf,   name="Random Forest", ax=plt.gca())
RocCurveDisplay.from_predictions(y_test, y_proba_gb,   name="Gradient Boosting", ax=plt.gca())
RocCurveDisplay.from_predictions(y_test, y_proba_mlp,  name="MLP", ax=plt.gca())
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("ROC Curves – Test Set")
plt.legend()
plt.show()

## 10. Baseline Feature Importance (Gini)

In [ ]:
preprocessor    = best_rf.named_steps["preprocess"]
rf_clf          = best_rf.named_steps["clf"]
cat_enc         = preprocessor.named_transformers_["cat"]
cat_names_out   = list(cat_enc.get_feature_names_out(categorical_features))
all_feat_names  = list(numeric_features) + cat_names_out

fi_df = pd.DataFrame({
    "feature"   : all_feat_names,
    "importance": rf_clf.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x="importance", y="feature", data=fi_df.head(15), orient="h")
plt.title("Top 15 RF Feature Importances (Gini – baseline)")
plt.tight_layout()
plt.show()

## 11. XAI – SHAP Setup

In [ ]:
X_train_t  = preprocessor.transform(X_train)
X_test_t   = preprocessor.transform(X_test)
X_train_df = pd.DataFrame(X_train_t, columns=all_feat_names)
X_test_df  = pd.DataFrame(X_test_t,  columns=all_feat_names)

print("Computing SHAP values (RF)...")
explainer_rf   = shap.TreeExplainer(rf_clf)
shap_values_rf = explainer_rf.shap_values(X_test_df)

if isinstance(shap_values_rf, list):
    sv_rf = shap_values_rf[1]
elif hasattr(shap_values_rf, 'shape') and len(shap_values_rf.shape) == 3:
    sv_rf = shap_values_rf[:, :, 1]
else:
    sv_rf = shap_values_rf

base_val = explainer_rf.expected_value[1] if hasattr(explainer_rf.expected_value, '__len__') else explainer_rf.expected_value

print(f"SHAP value matrix shape: {sv_rf.shape}")